In [4]:
import os
import glob
import pandas as pd
import re
import numpy as np

VALID_METHODS = {"Fast_Shap_1", "Fast_Shap_3", "Fast_Shap_5", "Shapley", "None"}

def parse_filename(filename: str) -> tuple | None:
    """
    Parse filenames like:
        sensitivity_fMRI_1_0_ResidualMLP_Fast_Shap_1.csv
    Returns: dataset, series, subject, model, method
    """
    name = os.path.splitext(os.path.basename(filename))[0]

    # Remove optional prefixes
    prefix_match = re.match(r"^(sensitivity)_", name)
    if prefix_match:
        name = name[prefix_match.end():]

    parts = name.split("_")
    if len(parts) < 5:
        return None

    dataset = parts[0]
    try:
        series = int(parts[1])
        subject = int(parts[2])
    except ValueError:
        return None

    model = parts[3]
    method = "_".join(parts[4:])

    if method not in VALID_METHODS:
        return None

    return dataset, series, subject, model, method


def collect_all_metrics(root_dir: str, dataset: str, series: int) -> pd.DataFrame:
    """
    Scan folder, parse filenames, load CSVs.
    Extracts the BEST value for each metric INDEPENDENTLY:
      - max_AUROC: Global max of the AUROC column.
      - max_AUPRC: Global max of the AUPRC column.
      - min_val_loss: Global min of the val_loss column.
    """
    pattern = os.path.join(root_dir, f"*{dataset}_{series}_*.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"[INFO] No files found in {root_dir} matching series {series}")
        return pd.DataFrame()

    records = []

    for fpath in files:
        parsed = parse_filename(fpath)
        if parsed is None:
            continue
        dataset_i, series_i, subject_i, model, method = parsed
        if series_i != series:
            continue

        # Load CSV
        try:
            df = pd.read_csv(fpath, on_bad_lines="skip")
        except Exception as e:
            print(f"[WARN] Failed to load {fpath}: {e}")
            continue
        if df.empty:
            continue

        # -----------------------------------------------------------
        # 1. Identify Columns
        # -----------------------------------------------------------
        # Metric columns
        auroc_col, auprc_col = None, None
        for col_pair in [(f"AUROC_{method}", f"AUPRC_{method}"), ("AUROC", "AUPRC")]:
            if col_pair[0] in df.columns and col_pair[1] in df.columns:
                auroc_col, auprc_col = col_pair
                break
        
        # Val loss column
        val_loss_col = None
        possible_loss_names = ["val_loss", "loss_val", "valid_loss", "Validation Loss"]
        for name in possible_loss_names:
            if name in df.columns:
                val_loss_col = name
                break

        if auroc_col is None:
            continue

        # -----------------------------------------------------------
        # 2. Extract Independent Bests
        # -----------------------------------------------------------
        # Ensure numeric types
        df[auroc_col] = pd.to_numeric(df[auroc_col], errors="coerce")
        df[auprc_col] = pd.to_numeric(df[auprc_col], errors="coerce")
        
        # Calculate Max AUROC / AUPRC
        # (We use max() on the whole column directly)
        max_auroc = df[auroc_col].max()
        max_auprc = df[auprc_col].max()
        
        # Calculate Min Val Loss
        min_val_loss = np.nan
        if val_loss_col:
            df[val_loss_col] = pd.to_numeric(df[val_loss_col], errors="coerce")
            min_val_loss = df[val_loss_col].min()

        # Skip if data is completely junk (all NaNs)
        if pd.isna(max_auroc):
            continue

        record = {
            "dataset": dataset_i,
            "series": series_i,
            "subject": subject_i,
            "model": model,
            "method": method,
            "max_AUROC": max_auroc,
            "max_AUPRC": max_auprc,
            "min_val_loss": min_val_loss
        }
        records.append(record)

    df_out = pd.DataFrame(records)
    return df_out


def summarize_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute mean/std across subjects for each (dataset, series, model, method)
    """
    if df.empty:
        return df

    grouped = (
        df.groupby(["method"])
        .agg(
            max_AUROC_mean=("max_AUROC", "mean"),
            max_AUROC_std=("max_AUROC", "std"),
            max_AUPRC_mean=("max_AUPRC", "mean"),
            max_AUPRC_std=("max_AUPRC", "std"),
            min_val_loss_mean=("min_val_loss", "mean"),
            min_val_loss_std=("min_val_loss", "std"),
            n_subjects=("subject", "nunique"),
        )
        .reset_index()
    )
    return grouped

# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    # Update this path
    root_dir = "/Users/merlyn/Documents/Projects/Empirical Granger Causality/Code/SRNGC/server_results/sensitivity_rand_proj"
    dataset = "fMRI"
    series = 1

    df_all = collect_all_metrics(root_dir, dataset, series)
    
    if not df_all.empty:
        summary = summarize_metrics(df_all)
        
        print("\nSummary over subjects:")
        pd.set_option('display.max_columns', None)
        print(summary)
    else:
        print("No valid data found.")


Summary over subjects:
        method  max_AUROC_mean  max_AUROC_std  max_AUPRC_mean  max_AUPRC_std  \
0  Fast_Shap_1        0.847822       0.022497        0.634066       0.036209   
1  Fast_Shap_3        0.848327       0.027940        0.639948       0.033239   
2  Fast_Shap_5        0.850979       0.021889        0.637849       0.042150   
3         None        0.670739       0.010170        0.328472       0.040196   
4      Shapley        0.856345       0.021524        0.676456       0.019346   

   min_val_loss_mean  min_val_loss_std  n_subjects  
0           0.963972          0.032089           5  
1           0.963075          0.037860           5  
2           0.960913          0.033666           5  
3           1.060134          0.038475           5  
4           0.946260          0.037556           5  
